In [1]:
# 02 - Visualize Augmentation & Features
#See what hybrid augmentation does and compare extracted features.

In [2]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from audiomentations import Compose, AddBackgroundNoise, PitchShift, TimeStretch
import sys
import os

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0,PROJECT_ROOT)

print("project root",PROJECT_ROOT);
print("src exists",os.path.exists(os.path.join(PROJECT_ROOT,"src")))

from src.feature_extraction import extract_log_mel, extract_mfcc, extract_gfcc
import torch
import random
import os
    
sr = 22050
    
# Load one clean sample
clean_path = '../data/clean_audio/positive/' + os.listdir('../data/clean_audio/positive')[0]
audio_clean, _ = librosa.load(clean_path, sr=sr)
    
# Load a random noise
noise_files = []
for root, _, files in os.walk('../data/noises'):
    for f in files:
        if f.endswith('.wav'):
            noise_files.append(os.path.join(root, f))
noise_path = random.choice(noise_files)
noise, _ = librosa.load(noise_path, sr=sr)
    
# Define augmentation
augment = Compose([
    AddBackgroundNoise(sounds_path='../data/noises/', min_snr_in_db=5, max_snr_in_db=20, p=1.0),
    PitchShift(min_semitones=-2, max_semitones=2, p=0.8),
    TimeStretch(min_rate=0.9, max_rate=1.1, p=0.8)
])
    
audio_aug = augment(samples=audio_clean.copy(), sample_rate=sr)

project root E:\SLIIT\Year_4\Semester_1\Research Project\RP-Project
src exists True


IndexError: list index out of range

In [ ]:
# Plot waveform
plt.figure(figsize=(14, 5))
librosa.display.waveshow(audio_clean, sr=sr, label='Clean', alpha=0.8)
librosa.display.waveshow(audio_aug, sr=sr, label='Augmented', alpha=0.6)
plt.legend()
plt.title('Waveform: Clean vs Augmented')
plt.show()

In [ ]:
# Extract and plot features
def plot_feature(audio, title):
    log_mel = extract_log_mel(audio, sr=sr).numpy()
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(log_mel, sr=sr, x_axis='time', y_axis='mel')
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.show()
    
plot_feature(audio_clean, 'Log-Mel Spectrogram - Clean')
plot_feature(audio_aug, 'Log-Mel Spectrogram - Augmented (Noisy + Pitch + Stretch)')